In [1]:
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
import os


C:\Users\mmunda\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
open_api_key = os.getenv("AZURE_OPENAI_API_KEY")
open_api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
open_api_version = os.getenv("AZURE_OPENAI_API_VERSION")

In [4]:
llm = AzureChatOpenAI(azure_endpoint=open_api_endpoint,
                          api_key=open_api_key,
                          azure_deployment="gpt-4o",
                          api_version= open_api_version)


In [5]:
from langchain_openai import AzureOpenAIEmbeddings
 
embedding = AzureOpenAIEmbeddings(
    api_key=open_api_key,
    azure_deployment="text-embedding-3-small",
    azure_endpoint=open_api_endpoint,
    api_version= open_api_version,
    dimensions=1024
)

In [6]:
pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_classic.document_loaders import PyMuPDFLoader

In [8]:
loader = PyMuPDFLoader(
    file_path="50_Laptop.pdf"
)
 
documents = loader.load()
print(documents)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-01T12:02:33+05:30', 'source': '50_Laptop.pdf', 'file_path': '50_Laptop.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Munda, Manas Ranjan', 'subject': '', 'keywords': '', 'moddate': '2026-04-01T12:02:33+05:30', 'trapped': '', 'modDate': "D:20260401120233+05'30'", 'creationDate': "D:20260401120233+05'30'", 'page': 0}, page_content='ID | Name | CPU | RAM | Storage | GPU | Price | Discount \n1 | ASUS Vivobook 15 | i5 13th Gen | 8GB | 512GB SSD | Integrated | 45999 | 25% \n2 | HP Victus 15 | i5 12th Gen | 16GB | 512GB SSD | RTX 3050 | 58990 | 20% \n3 | Lenovo LOQ | i5-12450HX | 16GB | 512GB SSD | RTX 4050 | 68940 | 18% \n4 | Dell Inspiron 15 | i5 13th Gen | 8GB | 512GB SSD | Integrated | 55000 | 15% \n5 | ASUS Vivobook Go 15 | Ryzen 5 | 8GB | 512GB SSD | Integrated | 37990 | 30% \n6 | HP Pavilion x360 | i5 13th Gen | 16GB | 512G

In [9]:
for i,k in enumerate(documents):
    print("------- Index: ",i, "----------")
    print(k.metadata)
    print(k.page_content[:200])

------- Index:  0 ----------
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-01T12:02:33+05:30', 'source': '50_Laptop.pdf', 'file_path': '50_Laptop.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Munda, Manas Ranjan', 'subject': '', 'keywords': '', 'moddate': '2026-04-01T12:02:33+05:30', 'trapped': '', 'modDate': "D:20260401120233+05'30'", 'creationDate': "D:20260401120233+05'30'", 'page': 0}
ID | Name | CPU | RAM | Storage | GPU | Price | Discount 
1 | ASUS Vivobook 15 | i5 13th Gen | 8GB | 512GB SSD | Integrated | 45999 | 25% 
2 | HP Victus 15 | i5 12th Gen | 16GB | 512GB SSD | RTX 3050 
------- Index:  1 ----------
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-01T12:02:33+05:30', 'source': '50_Laptop.pdf', 'file_path': '50_Laptop.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Munda, Manas Ra

# Chunking

In [10]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [11]:
textsplitters = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
)
 
docs = textsplitters.split_documents(documents=documents)

In [12]:
len(docs)

10

In [13]:
from langchain_classic.vectorstores import FAISS
 
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding
)

In [14]:
print("=== VECTORSTORE INSPECTION ===")
 
# Count docs
print("Total Document: ", len(vectorstore.docstore._dict))
print("Total Vector: ", vectorstore.index.ntotal)
 
 
# Show first 2 docs
for i, (k,v) in enumerate(vectorstore.docstore._dict.items()):
    if i >= 2:
        break
    print("\n--- Document", i, "---")
    print("ID:", k)
    print("Content:", v.page_content[:200], "...")
    print("Metadata:", v.metadata)
 
vec = vectorstore.index.reconstruct(0)
print("\nVector length:", len(vec))
print("Sample vector (first 10 dims):", vec[:10])

=== VECTORSTORE INSPECTION ===
Total Document:  10
Total Vector:  10

--- Document 0 ---
ID: ad5ab2bf-c09b-4009-a12f-926645e9d450
Content: ID | Name | CPU | RAM | Storage | GPU | Price | Discount 
1 | ASUS Vivobook 15 | i5 13th Gen | 8GB | 512GB SSD | Integrated | 45999 | 25% 
2 | HP Victus 15 | i5 12th Gen | 16GB | 512GB SSD | RTX 3050  ...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-01T12:02:33+05:30', 'source': '50_Laptop.pdf', 'file_path': '50_Laptop.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Munda, Manas Ranjan', 'subject': '', 'keywords': '', 'moddate': '2026-04-01T12:02:33+05:30', 'trapped': '', 'modDate': "D:20260401120233+05'30'", 'creationDate': "D:20260401120233+05'30'", 'page': 0}

--- Document 1 ---
ID: 29b5f998-6c4d-45cc-8c1f-65cdc0f6cdd3
Content: 5 | ASUS Vivobook Go 15 | Ryzen 5 | 8GB | 512GB SSD | Integrated | 37990 | 30% 
6 | HP Pavilion x360 | i5 13th Ge

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [16]:
from langchain_classic.chains import RetrievalQA
 
chains = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = retriever,
    return_source_documents = True
)

In [18]:
query = "Suggest me a best laptop within 80,0000 ?"
result = chains(query)
 
print("🧠 Answer:\n", result["result"])
print("\n📚 Source Chunks:")
for doc in result["source_documents"]:
    print("\n---")
    print(doc.page_content[:300], "...")  

🧠 Answer:
 Based on your budget of ₹80,000, here are the best laptops within the range:

1. **Dell G15 Gaming**  
   - **Processor**: i5 12th Gen  
   - **RAM**: 16GB  
   - **Storage**: 512GB SSD  
   - **Graphics**: RTX 3050  
   - **Price**: ₹79,990  
   - **Discount**: 20%

2. **Lenovo ThinkPad E14**  
   - **Processor**: i5 13th Gen  
   - **RAM**: 16GB  
   - **Storage**: 512GB SSD  
   - **Graphics**: Integrated  
   - **Price**: ₹79,990  
   - **Discount**: 18%

3. **Acer Nitro 5**  
   - **Processor**: i5 12th Gen  
   - **RAM**: 16GB  
   - **Storage**: 512GB SSD  
   - **Graphics**: RTX 3050  
   - **Price**: ₹69,999  
   - **Discount**: 28%  

4. **Lenovo Flex 5**  
   - **Processor**: Ryzen 5  
   - **RAM**: 16GB  
   - **Storage**: 512GB SSD  
   - **Graphics**: Integrated  
   - **Price**: ₹69,990  
   - **Discount**: 20%

5. **ASUS Vivobook S14**  
   - **Processor**: i5 13th Gen  
   - **RAM**: 16GB  
   - **Storage**: 512GB SSD  
   - **Graphics**: Integrated  
   - *